# Notebook 04 — Training Pipeline

## D-058 — Deterministic 512-token causal input/target packing

This notebook begins the training pipeline from the frozen artifacts produced by Notebooks 01–03.

For this first chunk, we resolve one question only:

> **How does the exact 20,000,000-token training corpus become deterministic 512-token causal input/target examples?**

The corpus contract remains unchanged: it contains exactly 20,000,000 tokenizer-produced token IDs. Packing is a separate training-pipeline contract.

### Selected production rule

Use **non-overlapping 512-token input blocks with stride 512**.  
For example index `k`:

- `start = k * 512`
- `x = tokens[start : start + 512]`
- `y = tokens[start + 1 : start + 513]`

Only examples for which **both `x` and `y` contain 512 real corpus tokens** are admitted.

Consequences:

- no padding
- no synthetic wraparound
- no duplicated tail targets
- no variable-length final example
- every admitted label is the actual next token in the canonical corpus
- document-boundary token ID `0` is treated like any other corpus token during packing; the boundary semantics were already established during corpus construction


### Why the 20M remainder is not the number of dropped prediction targets

The corpus length satisfies:

`20,000,000 = 39,062 × 512 + 256`

But causal supervision is shifted by one token. A 512-token input needs **513 corpus positions** to supply all 512 next-token labels.

The final full example therefore uses one token from the apparent 256-token arithmetic remainder as its final target. The true unused tail is **255 target positions**, not 256.


In [ ]:
CORPUS_TOKENS = 20_000_000
CONTEXT_LENGTH = 512
STRIDE = CONTEXT_LENGTH

# A full example requires 512 inputs plus one following token
# to provide the 512th next-token target.
NUM_FULL_EXAMPLES = (CORPUS_TOKENS - 1) // CONTEXT_LENGTH
PREDICTION_TARGETS = NUM_FULL_EXAMPLES * CONTEXT_LENGTH
DISTINCT_CORPUS_POSITIONS_USED = PREDICTION_TARGETS + 1
UNUSED_TAIL_TOKENS = CORPUS_TOKENS - DISTINCT_CORPUS_POSITIONS_USED
POSSIBLE_NEXT_TOKEN_TARGETS = CORPUS_TOKENS - 1
UNUSED_NEXT_TOKEN_TARGETS = POSSIBLE_NEXT_TOKEN_TARGETS - PREDICTION_TARGETS

print(f"Corpus tokens:                  {CORPUS_TOKENS:,}")
print(f"Context length:                 {CONTEXT_LENGTH:,}")
print(f"Stride:                         {STRIDE:,}")
print(f"Full causal examples:           {NUM_FULL_EXAMPLES:,}")
print(f"Prediction targets / epoch:     {PREDICTION_TARGETS:,}")
print(f"Distinct corpus positions used: {DISTINCT_CORPUS_POSITIONS_USED:,}")
print(f"Unused tail tokens:             {UNUSED_TAIL_TOKENS:,}")
print(f"Unused next-token targets:      {UNUSED_NEXT_TOKEN_TARGETS:,}")

assert NUM_FULL_EXAMPLES == 39_062
assert PREDICTION_TARGETS == 19_999_744
assert DISTINCT_CORPUS_POSITIONS_USED == 19_999_745
assert UNUSED_TAIL_TOKENS == 255
assert UNUSED_NEXT_TOKEN_TARGETS == 255


### Exact indexing contract

All indices below are zero-based corpus positions.

The first example is:

- inputs: positions `0 ... 511`
- targets: positions `1 ... 512`

The second example is:

- inputs: positions `512 ... 1023`
- targets: positions `513 ... 1024`

Notice that the last target of one block is the first input token of the next block. This is **not duplicate supervision**: each target position appears once.

The final full example is:

- inputs: positions `19,999,232 ... 19,999,743`
- targets: positions `19,999,233 ... 19,999,744`

The remaining corpus positions `19,999,745 ... 19,999,999` are the explicitly unused 255-token tail.


In [ ]:
def causal_block_bounds(example_index: int, context_length: int = 512):
    # Return half-open input and target bounds for one deterministic causal block.
    if example_index < 0:
        raise ValueError("example_index must be non-negative")

    start = example_index * context_length
    x_bounds = (start, start + context_length)
    y_bounds = (start + 1, start + context_length + 1)
    return x_bounds, y_bounds


first_x, first_y = causal_block_bounds(0)
second_x, second_y = causal_block_bounds(1)
last_x, last_y = causal_block_bounds(NUM_FULL_EXAMPLES - 1)

assert first_x == (0, 512)
assert first_y == (1, 513)

assert second_x == (512, 1024)
assert second_y == (513, 1025)

assert last_x == (19_999_232, 19_999_744)
assert last_y == (19_999_233, 19_999_745)

# The final admitted target is corpus position 19,999,744.
assert last_y[1] - 1 == 19_999_744

# Exactly 255 corpus positions remain after that target.
assert CORPUS_TOKENS - last_y[1] == 255

print("D-058 boundary arithmetic: PASS")


### Deterministic packing function

The function below is intentionally small. It accepts any one-dimensional token sequence and returns only complete causal blocks.

For the production corpus, it will create logical views equivalent to:

- `x.shape == (39_062, 512)`
- `y.shape == (39_062, 512)`

Later in Notebook 04, we will implement the dataset/dataloader path without needlessly materializing duplicate copies of the entire corpus.


In [ ]:
import numpy as np


def pack_full_causal_blocks(token_ids, context_length: int = 512):
    # Pack a 1D token stream into deterministic, non-overlapping causal examples.
    token_ids = np.asarray(token_ids)

    if token_ids.ndim != 1:
        raise ValueError("token_ids must be one-dimensional")
    if context_length <= 0:
        raise ValueError("context_length must be positive")
    if len(token_ids) <= context_length:
        return (
            np.empty((0, context_length), dtype=token_ids.dtype),
            np.empty((0, context_length), dtype=token_ids.dtype),
        )

    n_examples = (len(token_ids) - 1) // context_length
    n_targets = n_examples * context_length

    x = token_ids[:n_targets].reshape(n_examples, context_length)
    y = token_ids[1 : n_targets + 1].reshape(n_examples, context_length)

    return x, y


### Synthetic validation

A monotonic token stream makes indexing errors visible immediately. We validate:

1. shape
2. one-token causal shift
3. stride-512 block starts
4. continuity across block boundaries
5. every admitted target appears exactly once
6. the incomplete tail is excluded rather than padded, wrapped, or duplicated


In [ ]:
# Small, transparent example: C=8 and 20 corpus tokens.
toy = np.arange(20, dtype=np.int64)
toy_x, toy_y = pack_full_causal_blocks(toy, context_length=8)

expected_x = np.array([
    [0, 1, 2, 3, 4, 5, 6, 7],
    [8, 9, 10, 11, 12, 13, 14, 15],
])

expected_y = np.array([
    [1, 2, 3, 4, 5, 6, 7, 8],
    [9, 10, 11, 12, 13, 14, 15, 16],
])

assert np.array_equal(toy_x, expected_x)
assert np.array_equal(toy_y, expected_y)

# Within every block, labels are the next corpus token.
assert np.array_equal(toy_y[:, :-1], toy_x[:, 1:])

# Across blocks, the previous block's final target is the next block's first input.
assert np.array_equal(toy_y[:-1, -1], toy_x[1:, 0])

# Targets are exactly corpus positions 1..16, once each.
assert np.array_equal(toy_y.reshape(-1), np.arange(1, 17))

# Positions 17, 18, 19 form the explicitly unused tail.
assert np.array_equal(toy[17:], np.array([17, 18, 19]))

print("Synthetic causal packing validation: PASS")


## D-058 result

**Decision:** use stride-512, non-overlapping 512-token inputs and one-token-shifted 512-token targets; admit only complete input/target pairs.

For the exact 20M-token corpus:

| Quantity | Exact value |
|---|---:|
| Corpus tokens | 20,000,000 |
| Context length | 512 |
| Stride | 512 |
| Full examples | 39,062 |
| Prediction targets per epoch | 19,999,744 |
| Distinct corpus positions participating | 19,999,745 |
| Explicitly unused tail tokens | 255 |
| Possible corpus next-token transitions not trained | 255 |

This deliberately sacrifices only `255 / 19,999,999 ≈ 0.001275%` of possible corpus next-token targets in exchange for a simple, fixed-shape, padding-free, duplication-free training contract.

**Stop point:** D-058 is resolved and validated.  
The next Notebook 04 chunk should build the dataset/batching path from this contract.
